# Semana 3 · El clima detrás de los semestres malos
## Sesión 2 · Calidad de los datos de lluvia

In [1]:
import duckdb
con = duckdb.connect("../data/processed/agro.duckdb", read_only=True)

In [2]:
quality=con.sql("""
SELECT station,
       EXTRACT(year FROM date) AS year,
       COUNT(*)                AS days,
       ROUND(SUM(rain_mm))     AS total_mm
FROM rain
GROUP BY station, year
ORDER BY station, year
""").df()
quality

,station,year,days,total_mm
0,NUNCHIA [35210030],2019,76,1228.0
1,NUNCHIA [35210030],2020,144,2864.0
2,PAZ DE ARIPORO [36015010],2007,365,1791.0
3,PAZ DE ARIPORO [36015010],2008,366,1631.0
4,PAZ DE ARIPORO [36015010],2009,365,1966.0
...,...,...,...,...
57,TRINIDAD - AUT [35235040],2022,166,1561.0
58,TRINIDAD - AUT [35235040],2023,170,1504.0
59,TRINIDAD - AUT [35235040],2024,185,1800.0
60,TRINIDAD - AUT [35235040],2025,252,1692.0


In [3]:
quality.pivot(index="year", columns="station", values="days")

station,NUNCHIA [35210030],PAZ DE ARIPORO [36015010],SAN LUIS PALENQUE [35230010],TRINIDAD - AUT [35235040]
year,,,,
2007,NaN,365.0,166.0,157.0
2008,NaN,366.0,164.0,138.0
2009,NaN,365.0,162.0,151.0
2010,NaN,364.0,206.0,191.0
2011,NaN,365.0,180.0,127.0
2012,NaN,366.0,188.0,159.0
2013,NaN,365.0,177.0,139.0
2014,NaN,365.0,172.0,143.0
2015,NaN,365.0,167.0,139.0


In [4]:
quality.pivot(index="year", columns="station", values="total_mm")

station,NUNCHIA [35210030],PAZ DE ARIPORO [36015010],SAN LUIS PALENQUE [35230010],TRINIDAD - AUT [35235040]
year,,,,
2007,NaN,1791.0,1704.0,1936.0
2008,NaN,1631.0,1731.0,2201.0
2009,NaN,1966.0,1957.0,2204.0
2010,NaN,2330.0,2303.0,2567.0
2011,NaN,2492.0,2173.0,1787.0
2012,NaN,2182.0,2331.0,2313.0
2013,NaN,1525.0,1889.0,2053.0
2014,NaN,1748.0,2298.0,2031.0
2015,NaN,1961.0,1942.0,1733.0


### Hallazgos de calidad
- Nunchía se descarta: solo tiene datos de 2019 y 2020.
- Un día sin dato puede ser un día seco que no se anotó o un día que nadie midió. En San Luis de Palenque, hasta 2017 hay unos 165 días por año, pero el total anual se parece al de los años completos (2009: 1.957 mm en 162 días; 2019: 1.949 mm en 365 días), así que esos días eran casi todos secos. En Trinidad, en cambio, no hay ni un dato de mayo de 2011, en plena temporada de lluvias: ahí nadie midió.
- La estación principal es Paz de Ariporo: tiene la serie diaria completa de 2007 a 2023.

## Sesión 2 · El ONI por cosecha

In [5]:
con.sql("""
WITH monthly AS (
    SELECT year,
           month,
           make_date(year, month, 1) + INTERVAL 3 MONTH AS shifted,
           oni
    FROM oni
)
SELECT * FROM monthly WHERE year = 2011
""").df()



,year,month,shifted,oni
0,2011,1,2011-04-01,-1.29
1,2011,2,2011-05-01,-1.06
2,2011,3,2011-06-01,-0.85
3,2011,4,2011-07-01,-0.68
4,2011,5,2011-08-01,-0.50
5,2011,6,2011-09-01,-0.34
6,2011,7,2011-10-01,-0.37
7,2011,8,2011-11-01,-0.50
8,2011,9,2011-12-01,-0.76
9,2011,10,2012-01-01,-0.94


In [6]:
oni_period=con.sql("""
WITH monthly AS (
    SELECT year,
           month,
           make_date(year, month, 1) + INTERVAL 3 MONTH AS shifted,
           oni
    FROM oni
),
by_period AS (
    SELECT year,
           month,
           CAST(EXTRACT(year FROM shifted) AS INTEGER) || '-' ||
           CASE WHEN EXTRACT(month FROM shifted) <= 6 THEN 'I' ELSE 'II' END AS period,
           oni
    FROM monthly
)
SELECT period,
       COUNT(*)           AS months,
       ROUND(AVG(oni), 2) AS oni_avg
FROM by_period
WHERE period >= '2008-I'
GROUP BY period
ORDER BY period
""").df()
oni_period

,period,months,oni_avg
0,2008-I,6,-1.57
1,2008-II,6,-0.54
2,2009-I,6,-0.68
3,2009-II,6,0.29
4,2010-I,6,1.22
5,2010-II,6,-0.68
6,2011-I,6,-1.30
7,2011-II,6,-0.53
8,2012-I,6,-0.77
9,2012-II,6,0.09


In [7]:
rain_period = con.sql("""
WITH daily AS (
    SELECT station,
           date + INTERVAL 3 MONTH AS shifted,
           rain_mm
    FROM rain
),
by_period AS (
    SELECT station,
           CAST(EXTRACT(year FROM shifted) AS INTEGER) || '-' ||
           CASE WHEN EXTRACT(month FROM shifted) <= 6 THEN 'I' ELSE 'II' END AS period,
           rain_mm
    FROM daily
)
SELECT station,
       period,
       COUNT(*)            AS days,
       ROUND(SUM(rain_mm)) AS rain_mm
FROM by_period
WHERE period >= '2008-I'
GROUP BY station, period
ORDER BY station, period
""").df()
rain_period

,station,period,days,rain_mm
0,NUNCHIA [35210030],2019-II,41,700.0
1,NUNCHIA [35210030],2020-I,42,545.0
2,NUNCHIA [35210030],2020-II,103,2358.0
3,NUNCHIA [35210030],2021-I,34,489.0
4,PAZ DE ARIPORO [36015010],2008-I,183,251.0
...,...,...,...,...
110,TRINIDAD - AUT [35235040],2024-I,44,313.0
111,TRINIDAD - AUT [35235040],2024-II,124,1134.0
112,TRINIDAD - AUT [35235040],2025-I,140,790.0
113,TRINIDAD - AUT [35235040],2025-II,148,1259.0


In [8]:
rain_period.pivot(index="period", columns="station", values="days")

station,NUNCHIA [35210030],PAZ DE ARIPORO [36015010],SAN LUIS PALENQUE [35230010],TRINIDAD - AUT [35235040]
period,,,,
2008-I,NaN,183.0,42.0,33.0
2008-II,NaN,183.0,117.0,104.0
2009-I,NaN,182.0,56.0,48.0
2009-II,NaN,183.0,116.0,115.0
2010-I,NaN,182.0,55.0,41.0
2010-II,NaN,182.0,136.0,129.0
2011-I,NaN,182.0,60.0,51.0
2011-II,NaN,183.0,120.0,89.0
2012-I,NaN,183.0,68.0,43.0


In [9]:
rain_period.pivot(index="period", columns="station", values="rain_mm")

station,NUNCHIA [35210030],PAZ DE ARIPORO [36015010],SAN LUIS PALENQUE [35230010],TRINIDAD - AUT [35235040]
period,,,,
2008-I,NaN,251.0,450.0,551.0
2008-II,NaN,1241.0,1411.0,1768.0
2009-I,NaN,446.0,368.0,474.0
2009-II,NaN,1546.0,1574.0,1947.0
2010-I,NaN,468.0,457.0,421.0
2010-II,NaN,1769.0,1811.0,1891.0
2011-I,NaN,511.0,393.0,506.0
2011-II,NaN,1976.0,1759.0,1288.0
2012-I,NaN,623.0,479.0,617.0


In [10]:
con.sql("""
SELECT EXTRACT(month FROM date) AS month,
       COUNT(*)                 AS days,
       ROUND(SUM(rain_mm))      AS rain_mm
FROM rain
WHERE station LIKE 'PAZ%'
  AND date BETWEEN '2020-04-01' AND '2020-09-30'
GROUP BY month
ORDER BY month
""").df()

,month,days,rain_mm
0,6,30,216.0
1,7,31,331.0
2,8,31,189.0
3,9,30,135.0


### Hallazgos de la lluvia por cosecha
- En 2020-II, Paz de Ariporo suma 870 mm, unos 600 mm menos que un periodo II típico (1.486 mm en promedio en las ventanas completas). No fue una sequía: la ventana tiene 122 de sus 183 días porque faltan abril y mayo de 2020, cuando empiezan las lluvias y se siembra. Los meses que sí tienen dato llovieron normal o más: julio tuvo 331 mm, frente a 265 mm en promedio entre 2008 y 2023.
- Regla: solo se confía en una ventana que tenga casi todos sus días. A Paz de Ariporo también le faltan días en 2020-I, 2024-II, 2025-I, 2025-II y 2026-I.